In [ ]:
import pandas as pd

# ================================
# 1. Load datasets
# ================================

merged = pd.read_csv("/content/sample_data/merged_dataset.tsv", sep="\t")

informative = pd.read_csv(
    "/content/sample_data/task_informative_text_img_full.tsv", sep="\t"
)
humanitarian = pd.read_csv(
    "/content/sample_data/task_humanitarian_text_img_full.tsv", sep="\t"
)
damage = pd.read_csv(
    "/content/sample_data/task_damage_text_img_full.tsv", sep="\t"
)

# ================================
# 2. Create unique sample IDs
# ================================
# Each image is a distinct sample

def make_sample_id(df):
    return df["image_id"].astype(str)

merged["sample_id"] = make_sample_id(merged)
informative["sample_id"] = make_sample_id(informative)
humanitarian["sample_id"] = make_sample_id(humanitarian)
damage["sample_id"] = make_sample_id(damage)

# ================================
# 3. Build split maps from originals
# ================================

info_split_map = dict(zip(informative["sample_id"], informative["split"]))
damage_split_map = dict(zip(damage["sample_id"], damage["split"]))
human_split_map = dict(zip(humanitarian["sample_id"], humanitarian["split"]))

# ================================
# 4. Assign global splits (priority-based)
# ================================

def assign_split(row):
    sid = row["sample_id"]

    # Priority 1: Informativeness (anchor task)
    if sid in info_split_map:
        return info_split_map[sid]

    # Priority 2: Damage
    if sid in damage_split_map:
        return damage_split_map[sid]

    # Priority 3: Humanitarian
    if sid in human_split_map:
        return human_split_map[sid]

    raise ValueError(f"Unassigned sample: {sid}")

merged["split"] = merged.apply(assign_split, axis=1)

# ================================
# 5. Sanity checks
# ================================

assert merged["split"].isna().sum() == 0, "Some samples were not assigned a split"

print("Global split counts:")
print(merged["split"].value_counts())
print()

# ================================
# 6. Define FINAL column structure
# ================================

FINAL_COLUMNS = [
    "event_name",
    "tweet_id",
    "image_id",
    "tweet_text",
    "image",
    "label_task1",
    "label_task2",
    "label_task3",
    "event_category",
]

# Verify all required columns exist
missing_cols = set(FINAL_COLUMNS) - set(merged.columns)
assert not missing_cols, f"Missing columns in merged dataset: {missing_cols}"

# ================================
# 7. Save final subsets (NO split column)
# ================================

for split_name in ["train", "dev", "test"]:
    subset = merged[merged["split"] == split_name][FINAL_COLUMNS]

    subset.to_csv(
        f"/content/sample_data/merged_{split_name}.tsv",
        sep="\t",
        index=False
    )

print("✅ Final datasets saved:")
print(" - merged_train.tsv")
print(" - merged_dev.tsv")
print(" - merged_test.tsv")
